# DS317_05 — Trích xuất Bảng Thời tiết NOAA RTMA 2025 Gắn Khóa Ghép nối (16 Cột Sidecar)

**Mục tiêu:** Thay vì nhân bản toàn bộ `27` cột AIS của `03_ban_tien_xu_ly_27_thuoc_tinh` thành Bản 39 cột (~dễ tràn đĩa `/kaggle/working`), notebook xuất ra **bảng thời tiết sidecar gọn nhẹ (`weather_rtma_2025_sidecar/{train,test}.parquet`)** gồm **16 cột**: **4 cột khóa khớp 1-1 với Bản 3** (`row_id`, `mmsi`, `trip_id`, `base_date_time`) + **12 cột thời tiết NOAA RTMA 2025**.

## Kiến trúc & Quy chuẩn Thiết kế
1. **Giảm >60% dung lượng đĩa & RAM đọc/ghi:** Chỉ đọc 7 cột cần thiết từ Bản 3 (`mmsi`, `trip_id`, `base_date_time`, `latitude`, `longitude`, `cog_sin`, `cog_cos`), không lưu trùng 24 cột AIS còn lại, và dọn shard ngay sau mỗi split (`train` rồi `test`) để tránh tồn đồng thời 2 bộ shard.
2. **Khớp khóa an toàn với Bản 3:** Cặp khóa nghiệp vụ `("mmsi", "trip_id", "base_date_time")` trùng tên và kiểu dữ liệu với Bản 3, kèm khóa định vị `row_id` (`int64`) bảo đảm ghép nối 1-1 không nhân bản dòng nếu AIS có timestamp trùng.
3. **Nguồn dữ liệu thời tiết:** Amazon S3 công khai `https://noaa-rtma-pds.s3.amazonaws.com` (4 miền CONUS & OCONUS: `rtma2p5`, `akrtma`, `hirtma`, `prrtma`), chu kỳ `1 giờ/lần` năm 2025, giải mã GRIB2 trực tiếp trong RAM bằng `.idx` + HTTP Range + `eccodes`.
4. **Khớp thời gian chống rò rỉ (Zero-Lookahead):** Dùng `base_date_time.floor('1h')` UTC cho cả `train` và `test`; kèm hàm `resolve_rtma_inference_hour` mô phỏng cơ chế chờ bản tin RTMA thực tế (~15-25 phút) khi suy luận online.
5. **Chuẩn hóa đơn vị & Gió tương đối (Lựa chọn B):** `rtma_TMP_C`, `rtma_DPT_C` (`°C`), `rtma_PRES_hpa` (`hPa`), `rtma_VIS_nm` (`NM`), `rtma_WIND_knots`, `rtma_GUST_knots` (`knots`), `rtma_wdir_sin`, `rtma_wdir_cos`, `rtma_headwind_knots`, `rtma_crosswind_knots`, giữ nguyên `NaN` + `rtma_domain` (`0..4`) + `weather_missing_flag` (`0`/`1`).

In [ ]:
"""Cell 1: Cài đặt thư viện, cấu hình hằng số và Sidecar 16-column Schema."""

from concurrent.futures import ThreadPoolExecutor, as_completed
import datetime as dt
import glob
import math
import os
import shutil
import struct
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.request

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


def ensure_eccodes() -> object:
    """Đảm bảo thư viện eccodes khả dụng để giải mã GRIB2.

    Returns:
        object: Module eccodes đã được import.
    """
    try:
        import eccodes
        return eccodes
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "eccodes"]
        )
        import eccodes
        return eccodes


IS_KAGGLE = os.path.exists("/kaggle")
WORK_DIR = "/kaggle/working" if IS_KAGGLE else "./kaggle_work_rtma"
os.makedirs(WORK_DIR, exist_ok=True)

INPUT_PREP_DIR = os.path.join(WORK_DIR, "03_ban_tien_xu_ly_27_thuoc_tinh")
OUTPUT_WEATHER_DIR = os.path.join(WORK_DIR, "weather_rtma_2025_sidecar")
SHARD_TMP_DIR = os.path.join(WORK_DIR, "tmp_rtma_shards")
os.makedirs(OUTPUT_WEATHER_DIR, exist_ok=True)
os.makedirs(SHARD_TMP_DIR, exist_ok=True)

# Đặt 24 để chạy thử trước khi chạy full 365 ngày (None) trên Kaggle
SMOKE_TEST_HOURS = None
TARGET_MONTHS = None
MAX_HTTP_WORKERS = 8
HTTP_TIMEOUT_SEC = 30
MAX_RETRIES = 3

# Hằng số chuyển đổi đơn vị đồng bộ chuẩn hàng hải
R_EARTH_DEFAULT_M = 6_371_200.0
KELVIN_OFFSET = 273.15
MS_TO_KNOTS = 3600.0 / 1852.0
METERS_TO_NM = 1.0 / 1852.0
PA_TO_HPA = 0.01

S3_BASE_URL = "https://noaa-rtma-pds.s3.amazonaws.com"
TARGET_VARS = ("PRES", "TMP", "DPT", "WDIR", "WIND", "GUST", "VIS")

# Thứ tự ưu tiên OCONUS phân giải cao trước CONUS
DOMAIN_PRIORITY = (
    ("prrtma", 4, "{d}.{ymd}/{d}.t{hh}z.2dvaranl_ndfd.grb2"),
    ("hirtma", 3, "{d}.{ymd}/{d}.t{hh}z.2dvaranl_ndfd.grb2"),
    ("akrtma", 2, "{d}.{ymd}/{d}.t{hh}z.2dvaranl_ndfd_3p0.grb2"),
    ("rtma2p5", 1, "{d}.{ymd}/{d}.t{hh}z.2dvaranl_ndfd.grb2_wexp"),
)

COLS_AIS_NEEDED = (
    "mmsi",
    "trip_id",
    "base_date_time",
    "latitude",
    "longitude",
    "cog_sin",
    "cog_cos",
)

# Schema 16 cột: 4 cột khóa khớp Bản 3 (27 cột) + 12 cột RTMA 2025
SCHEMA_WEATHER_SIDECAR = pa.schema([
    ("row_id", pa.int64()),
    ("mmsi", pa.int64()),
    ("trip_id", pa.string()),
    ("base_date_time", pa.timestamp("us")),
    ("rtma_domain", pa.int8()),
    ("weather_missing_flag", pa.int8()),
    ("rtma_TMP_C", pa.float32()),
    ("rtma_DPT_C", pa.float32()),
    ("rtma_PRES_hpa", pa.float32()),
    ("rtma_VIS_nm", pa.float32()),
    ("rtma_WIND_knots", pa.float32()),
    ("rtma_GUST_knots", pa.float32()),
    ("rtma_wdir_sin", pa.float32()),
    ("rtma_wdir_cos", pa.float32()),
    ("rtma_headwind_knots", pa.float32()),
    ("rtma_crosswind_knots", pa.float32()),
])

COLS_KEY_JOIN = ("row_id", "mmsi", "trip_id", "base_date_time")
COLS_WEATHER_12 = [f.name for f in SCHEMA_WEATHER_SIDECAR][4:]
COLS_WEATHER_SIDECAR = [f.name for f in SCHEMA_WEATHER_SIDECAR]

In [ ]:
"""Cell 2: Dò tìm tập 27 cột, đọc 7 cột cần thiết và hàm chờ."""


def find_split_path(split_name: str) -> str:
    """Tìm tệp train.parquet hoặc test.parquet của Bản 3 (27 cột).

    Args:
        split_name (str): Tên phân vùng ('train' hoặc 'test').

    Returns:
        str: Đường dẫn tới tệp Parquet trên Kaggle hoặc cục bộ.
    """
    fname = f"{split_name}.parquet"
    local_path = os.path.join(INPUT_PREP_DIR, fname)
    if os.path.exists(local_path):
        return local_path

    if os.path.exists("/kaggle/input"):
        patterns = [
            f"/kaggle/input/**/*27*{fname}",
            f"/kaggle/input/**/*tien_xu_ly*/{fname}",
            f"/kaggle/input/**/{fname}",
        ]
        for pat in patterns:
            hits = sorted(glob.glob(pat, recursive=True))
            for path in hits:
                cols = pq.read_schema(path).names
                if (
                    "cog_sin" in cols
                    and "imputation_flag" in cols
                    and "rtma_domain" not in cols
                ):
                    return path
    return local_path


def normalize_ais_input_batch(
    table: pa.Table, start_row_id: int = 0
) -> pd.DataFrame:
    """Chuẩn hóa batch AIS gọn nhẹ chỉ giữ 7 cột cần + row_id.

    Args:
        table (pa.Table): Batch dữ liệu đọc từ Bản 3 Parquet.
        start_row_id (int): Chỉ số dòng bắt đầu trong tệp gốc.

    Returns:
        pd.DataFrame: DataFrame có 'row_id' và 7 cột cần thiết.
    """
    missing_cols = [c for c in COLS_AIS_NEEDED if c not in table.column_names]
    if missing_cols:
        raise ValueError(
            f"Thiếu cột bắt buộc từ Bản 3 (27 cột): {missing_cols}"
        )
    df = table.select(list(COLS_AIS_NEEDED)).to_pandas()
    bdt = pd.to_datetime(df["base_date_time"], errors="coerce")
    if getattr(bdt.dt, "tz", None) is not None:
        bdt = bdt.dt.tz_convert(None)
    df["base_date_time"] = bdt
    df["row_id"] = np.arange(
        start_row_id, start_row_id + len(df), dtype=np.int64
    )
    return df


def resolve_rtma_inference_hour(
    obs_time_utc: dt.datetime,
    now_utc: dt.datetime,
    max_wait_minutes: int = 28,
) -> dict:
    """Xác định khung giờ RTMA khi suy luận realtime (đợi bản tin).

    Khi vận hành thực tế, bản tin tXXz thường phát hành sau mốc giờ XX
    khoảng 15-25 phút. Hàm tính mốc floor('1h') tương ứng và cho biết
    hệ thống cần đợi bao lâu để lấy đủ dữ liệu thời tiết trước predict.

    Args:
        obs_time_utc (datetime): Thời điểm AIS cần dự báo (UTC).
        now_utc (datetime): Thời gian hệ thống hiện tại (UTC).
        max_wait_minutes (int): Ngưỡng chờ tối đa sau đầu giờ (phút).

    Returns:
        dict: Gồm 'target_hour_utc', 'should_wait', 'wait_seconds' và
            'fallback_to_missing'.
    """
    target_hour = obs_time_utc.replace(minute=0, second=0, microsecond=0)
    elapsed_sec = (now_utc - target_hour).total_seconds()
    limit_sec = max_wait_minutes * 60.0
    typical_ready_sec = 25 * 60.0

    if elapsed_sec < typical_ready_sec:
        return {
            "target_hour_utc": target_hour,
            "should_wait": True,
            "wait_seconds": max(0.0, typical_ready_sec - elapsed_sec),
            "fallback_to_missing": elapsed_sec > limit_sec,
        }
    return {
        "target_hour_utc": target_hour,
        "should_wait": False,
        "wait_seconds": 0.0,
        "fallback_to_missing": False,
    }

In [ ]:
"""Cell 3: Phân tích GRIB2 Section 3 & Chiếu tọa độ giải tích."""


def _signed_int32(buf: bytes, offset: int) -> int:
    """Giải mã số nguyên có dấu bit cao nhất theo chuẩn WMO GRIB2."""
    val = struct.unpack(">I", buf[offset:offset + 4])[0]
    return -(val & 0x7FFFFFFF) if (val & 0x80000000) else val


def parse_grib2_gds(buf: bytes) -> dict:
    """Đọc Section 3 (Grid Definition Section) từ byte GRIB2.

    Args:
        buf (bytes): Phần đầu bản tin GRIB2 (tối thiểu 256 bytes).

    Returns:
        dict: Từ điển tham số chiếu (gdt, nx, ny, la1, lo1, dx_m...).
    """
    if buf[:4] != b"GRIB":
        raise ValueError("Không đúng định dạng GRIB2 header")

    pos = 16
    while pos < len(buf) - 5:
        sec_len = struct.unpack(">I", buf[pos:pos + 4])[0]
        sec_num = buf[pos + 4]
        if sec_num == 3:
            sec = buf[pos:pos + sec_len]
            gdt = struct.unpack(">H", sec[12:14])[0]
            sf_r, sv_r = struct.unpack(">Bi", sec[15:20])
            radius = sv_r * (10.0 ** (-sf_r))
            if radius <= 0.0:
                radius = R_EARTH_DEFAULT_M

            nx, ny = struct.unpack(">II", sec[30:38])
            info = {
                "gdt": gdt,
                "radius_m": float(radius),
                "nx": int(nx),
                "ny": int(ny),
                "la1": _signed_int32(sec, 38) / 1e6,
                "lo1": _signed_int32(sec, 42) / 1e6,
            }
            if gdt == 30:
                info.update({
                    "lad": _signed_int32(sec, 47) / 1e6,
                    "lov": _signed_int32(sec, 51) / 1e6,
                    "dx_m": struct.unpack(">I", sec[55:59])[0] / 1e3,
                    "dy_m": struct.unpack(">I", sec[59:63])[0] / 1e3,
                    "scan_mode": int(sec[64]),
                    "latin1": _signed_int32(sec, 65) / 1e6,
                    "latin2": _signed_int32(sec, 69) / 1e6,
                })
            elif gdt == 20:
                info.update({
                    "lad": _signed_int32(sec, 47) / 1e6,
                    "lov": _signed_int32(sec, 51) / 1e6,
                    "dx_m": struct.unpack(">I", sec[55:59])[0] / 1e3,
                    "dy_m": struct.unpack(">I", sec[59:63])[0] / 1e3,
                    "scan_mode": int(sec[64]),
                })
            elif gdt == 10:
                info.update({
                    "lad": _signed_int32(sec, 47) / 1e6,
                    "la2": _signed_int32(sec, 51) / 1e6,
                    "lo2": _signed_int32(sec, 55) / 1e6,
                    "scan_mode": int(sec[59]),
                    "dx_m": struct.unpack(">I", sec[64:68])[0] / 1e3,
                    "dy_m": struct.unpack(">I", sec[68:72])[0] / 1e3,
                })
            return info
        pos += sec_len
    raise ValueError("Không tìm thấy Section 3 trong buffer GRIB2")


def _wrap_dlon(lam_rad: np.ndarray, lam0_rad: float) -> np.ndarray:
    """Chuẩn hóa chênh lệch kinh độ về [-pi, pi) an toàn 180°."""
    return np.mod(lam_rad - lam0_rad + math.pi, 2.0 * math.pi) - math.pi


def latlon_to_grid_ij(
    lat_deg: np.ndarray, lon_deg: np.ndarray, gds: dict
) -> tuple[np.ndarray, np.ndarray]:
    """Chiếu vector tọa độ (lat, lon) sang chỉ số lưới (fx, fy) GRIB2.

    Args:
        lat_deg (np.ndarray): Mảng vĩ độ thập phân.
        lon_deg (np.ndarray): Mảng kinh độ thập phân.
        gds (dict): Tham số Section 3 đọc từ GRIB2.

    Returns:
        tuple[np.ndarray, np.ndarray]: Cặp tọa độ liên tục (fi, fj).
    """
    radius = gds.get("radius_m", R_EARTH_DEFAULT_M)
    gdt = gds["gdt"]
    phi = np.radians(np.asarray(lat_deg, dtype=np.float64))
    lam = np.radians(np.asarray(lon_deg, dtype=np.float64))
    phi_s = math.radians(gds["la1"])
    lam_s = math.radians(gds["lo1"])

    if gdt == 30:
        phi1 = math.radians(gds["latin1"])
        phi2 = math.radians(gds["latin2"])
        lam0 = math.radians(gds["lov"])
        if abs(phi1 - phi2) < 1e-10:
            cone = math.sin(phi1)
        else:
            num = math.log(math.cos(phi1) / math.cos(phi2))
            t1 = math.tan(math.pi / 4.0 + phi1 / 2.0)
            t2 = math.tan(math.pi / 4.0 + phi2 / 2.0)
            cone = num / math.log(t2 / t1)
        factor = (
            math.cos(phi1)
            * (math.tan(math.pi / 4.0 + phi1 / 2.0) ** cone)
            / cone
        )
        rho_s = radius * factor / (
            math.tan(math.pi / 4.0 + phi_s / 2.0) ** cone
        )
        dlon_s = ((lam_s - lam0 + math.pi) % (2.0 * math.pi)) - math.pi
        x_s = rho_s * math.sin(cone * dlon_s)
        y_s = -rho_s * math.cos(cone * dlon_s)

        rho = radius * factor / np.power(
            np.tan(math.pi / 4.0 + phi / 2.0), cone
        )
        dlon = _wrap_dlon(lam, lam0)
        x_val = rho * np.sin(cone * dlon)
        y_val = -rho * np.cos(cone * dlon)
        return (x_val - x_s) / gds["dx_m"], (y_val - y_s) / gds["dy_m"]

    if gdt == 20:
        phi_d = math.radians(gds["lad"])
        lam_v = math.radians(gds["lov"])
        scale = 2.0 * radius * (1.0 + math.sin(phi_d)) / 2.0
        rho_s = scale * math.tan(math.pi / 4.0 - phi_s / 2.0)
        dlon_s = ((lam_s - lam_v + math.pi) % (2.0 * math.pi)) - math.pi
        x_s = rho_s * math.sin(dlon_s)
        y_s = -rho_s * math.cos(dlon_s)

        rho = scale * np.tan(math.pi / 4.0 - phi / 2.0)
        dlon = _wrap_dlon(lam, lam_v)
        x_val = rho * np.sin(dlon)
        y_val = -rho * np.cos(dlon)
        return (x_val - x_s) / gds["dx_m"], (y_val - y_s) / gds["dy_m"]

    if gdt == 10:
        phi_d = math.radians(gds["lad"])
        coef = radius * math.cos(phi_d)
        y_s = coef * math.log(math.tan(math.pi / 4.0 + phi_s / 2.0))
        dlon = _wrap_dlon(lam, lam_s)
        x_val = coef * dlon
        y_val = coef * np.log(np.tan(math.pi / 4.0 + phi / 2.0))
        return x_val / gds["dx_m"], (y_val - y_s) / gds["dy_m"]

    raise ValueError(f"Chưa hỗ trợ Grid Definition Template 3.{gdt}")


def assign_rtma_domain_and_cells(
    lat_deg: np.ndarray, lon_deg: np.ndarray, domain_gds: dict
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Gán ID miền RTMA (0..4) và chỉ số ô lưới (gy, gx) gần nhất.

    Args:
        lat_deg (np.ndarray): Mảng vĩ độ tàu.
        lon_deg (np.ndarray): Mảng kinh độ tàu.
        domain_gds (dict): Ánh xạ domain_name -> tham số Section 3.

    Returns:
        tuple: (domain_ids [int8], grid_y [int32], grid_x [int32]).
    """
    n_rows = len(lat_deg)
    dom_ids = np.zeros(n_rows, dtype=np.int8)
    grid_y = np.full(n_rows, -1, dtype=np.int32)
    grid_x = np.full(n_rows, -1, dtype=np.int32)

    valid_geo = (
        np.isfinite(lat_deg)
        & np.isfinite(lon_deg)
        & (lat_deg >= 15.0)
        & (lat_deg <= 75.0)
        & (lon_deg >= -180.0)
        & (lon_deg <= -65.0)
    )

    for dom_name, dom_id, _ in DOMAIN_PRIORITY:
        gds = domain_gds.get(dom_name)
        if gds is None:
            continue
        unassigned = valid_geo & (dom_ids == 0)
        if not np.any(unassigned):
            break

        fi, fj = latlon_to_grid_ij(
            lat_deg[unassigned], lon_deg[unassigned], gds
        )
        in_box = (
            (fi >= -0.5)
            & (fi <= gds["nx"] - 0.5)
            & (fj >= -0.5)
            & (fj <= gds["ny"] - 0.5)
        )
        if not np.any(in_box):
            continue

        idx_sub = np.flatnonzero(unassigned)[in_box]
        ix = np.clip(np.rint(fi[in_box]).astype(np.int32), 0, gds["nx"] - 1)
        iy = np.clip(np.rint(fj[in_box]).astype(np.int32), 0, gds["ny"] - 1)
        dom_ids[idx_sub] = dom_id
        grid_x[idx_sub] = ix
        grid_y[idx_sub] = iy

    return dom_ids, grid_y, grid_x


In [ ]:
"""Cell 4: Tải .idx + HTTP Byte-Range gộp đoạn & Giải mã GRIB2 RAM."""


def http_get_bytes(url: str, byte_range: str = None) -> bytes:
    """Tải nội dung HTTP/HTTPS với cơ chế retry và optional Range.

    Args:
        url (str): Đường dẫn tệp trên Amazon S3.
        byte_range (str): Chuỗi header Range ('bytes=a-b') hoặc None.

    Returns:
        bytes: Dữ liệu nhị phân tải về.
    """
    headers = {}
    if byte_range is not None:
        headers["Range"] = byte_range

    last_err = None
    for attempt in range(MAX_RETRIES):
        req = urllib.request.Request(url, headers=headers)
        try:
            with urllib.request.urlopen(req, timeout=HTTP_TIMEOUT_SEC) as r:
                return r.read()
        except urllib.error.HTTPError as exc:
            if exc.code in (403, 404, 416):
                raise FileNotFoundError(
                    f"Không tồn tại trên S3 ({exc.code}): {url}"
                ) from exc
            last_err = exc
        except Exception as exc:
            last_err = exc
        time.sleep(0.5 * (2 ** attempt))
    raise RuntimeError(f"Lỗi HTTP sau {MAX_RETRIES} lần: {url}") from last_err


def parse_idx_byte_ranges(
    idx_text: str, wanted_vars: tuple = TARGET_VARS
) -> list[dict]:
    """Phân tích tệp .idx thành các dải byte và gộp đoạn liền kề.

    Args:
        idx_text (str): Nội dung text của tệp .idx từ NOAA S3.
        wanted_vars (tuple): Danh sách mã biến cần trích xuất.

    Returns:
        list[dict]: Các nhóm Range liên tiếp gồm 'start', 'end' và
            danh sách thông tin biến 'items'.
    """
    records = []
    for raw_line in idx_text.strip().splitlines():
        parts = raw_line.split(":")
        if len(parts) < 5:
            continue
        records.append({
            "msg_id": int(parts[0]),
            "start": int(parts[1]),
            "var": parts[3].strip(),
            "level": parts[4].strip(),
        })

    for idx, rec in enumerate(records):
        next_start = (
            records[idx + 1]["start"] - 1
            if idx + 1 < len(records)
            else None
        )
        rec["end"] = next_start

    wanted_set = set(wanted_vars)
    selected = []
    seen_vars = set()
    for rec in records:
        if rec["var"] in wanted_set and rec["var"] not in seen_vars:
            selected.append(rec)
            seen_vars.add(rec["var"])

    if not selected:
        return []

    groups = []
    cur = {
        "start": selected[0]["start"],
        "end": selected[0]["end"],
        "items": [selected[0]],
    }
    for rec in selected[1:]:
        is_contiguous = (
            cur["end"] is not None and rec["start"] == cur["end"] + 1
        )
        if is_contiguous:
            cur["end"] = rec["end"]
            cur["items"].append(rec)
        else:
            groups.append(cur)
            cur = {
                "start": rec["start"],
                "end": rec["end"],
                "items": [rec],
            }
    groups.append(cur)
    return groups


def discover_domain_gds(sample_hour: dt.datetime = None) -> dict:
    """Tải 512 bytes đầu tiên mỗi miền từ S3 để đọc Section 3 chính xác.

    Args:
        sample_hour (datetime): Khung giờ mẫu trong năm 2025.

    Returns:
        dict: Từ điển domain_name -> GDS dictionary.
    """
    if sample_hour is None:
        sample_hour = dt.datetime(2025, 7, 1, 0, 0)
    ymd = sample_hour.strftime("%Y%m%d")
    hh = sample_hour.strftime("%H")

    out = {}
    for dom_name, _, tpl in DOMAIN_PRIORITY:
        key = tpl.format(d=dom_name, ymd=ymd, hh=hh)
        url = f"{S3_BASE_URL}/{key}"
        buf = http_get_bytes(url, byte_range="bytes=0-511")
        out[dom_name] = parse_grib2_gds(buf)
    return out


def fetch_hour_domain_arrays(
    eccodes_mod: object,
    hour_utc: dt.datetime,
    dom_name: str,
    tpl: str,
) -> dict | None:
    """Tải và giải mã 7 biến bề mặt của một (giờ, miền) trong RAM.

    Args:
        eccodes_mod (object): Module eccodes.
        hour_utc (datetime): Mốc giờ UTC đã làm tròn floor('1h').
        dom_name (str): Tên miền ('rtma2p5', 'akrtma', ...).
        tpl (str): Khuôn S3 key.

    Returns:
        dict | None: Dict var_name -> 2D float32 ndarray, hoặc None nếu 404.
    """
    ymd = hour_utc.strftime("%Y%m%d")
    hh = hour_utc.strftime("%H")
    key = tpl.format(d=dom_name, ymd=ymd, hh=hh)
    grb_url = f"{S3_BASE_URL}/{key}"
    idx_url = f"{grb_url}.idx"

    try:
        idx_text = http_get_bytes(idx_url).decode("utf-8", errors="replace")
    except FileNotFoundError:
        return None

    groups = parse_idx_byte_ranges(idx_text, TARGET_VARS)
    if not groups:
        return None

    arrays = {}
    for grp in groups:
        end_str = "" if grp["end"] is None else str(grp["end"])
        rng_hdr = f"bytes={grp['start']}-{end_str}"
        try:
            blob = http_get_bytes(grb_url, byte_range=rng_hdr)
        except FileNotFoundError:
            return None

        for item in grp["items"]:
            rel_start = item["start"] - grp["start"]
            if rel_start + 16 > len(blob):
                continue
            if blob[rel_start:rel_start + 4] != b"GRIB":
                continue
            msg_len = struct.unpack(
                ">Q", blob[rel_start + 8:rel_start + 16]
            )[0]
            msg_bytes = blob[rel_start:rel_start + msg_len]
            gid = eccodes_mod.codes_new_from_message(msg_bytes)
            try:
                ni = int(eccodes_mod.codes_get(gid, "Ni"))
                nj = int(eccodes_mod.codes_get(gid, "Nj"))
                miss = float(eccodes_mod.codes_get(gid, "missingValue"))
                vals = eccodes_mod.codes_get_values(gid).astype(np.float32)
                vals[vals == miss] = np.nan
                vals[np.abs(vals) >= 9.9e19] = np.nan
                arrays[item["var"]] = vals.reshape((nj, ni))
            finally:
                eccodes_mod.codes_release(gid)

    return arrays if arrays else None


In [ ]:
"""Cell 5: Chuyển đổi đơn vị, tính gió tương đối, hàm ghép nối & Kiểm thử."""


def extract_and_engineer_weather(
    df_ais: pd.DataFrame,
    dom_ids: np.ndarray,
    grid_y: np.ndarray,
    grid_x: np.ndarray,
    weather_cache: dict,
) -> pd.DataFrame:
    """Trích xuất 12 cột RTMA gắn khóa trùng Bản 3 (không nhân bản cột AIS).

    Args:
        df_ais (pd.DataFrame): Batch AIS gồm 'row_id' và 7 cột cần.
        dom_ids (np.ndarray): ID miền RTMA (0..4).
        grid_y (np.ndarray): Chỉ số hàng lưới y.
        grid_x (np.ndarray): Chỉ số cột lưới x.
        weather_cache (dict): Cache (hour_int, dom_id) -> dict arrays.

    Returns:
        pd.DataFrame: DataFrame 16 cột theo SCHEMA_WEATHER_SIDECAR.
    """
    n_rows = len(df_ais)
    raw = {
        v: np.full(n_rows, np.nan, dtype=np.float32) for v in TARGET_VARS
    }

    hours_ns = df_ais["base_date_time"].dt.floor("1h").to_numpy()
    pair_codes = (
        hours_ns.astype("datetime64[h]").astype(np.int64) * 10
        + dom_ids.astype(np.int64)
    )
    order = np.argsort(pair_codes, kind="mergesort")
    sorted_codes = pair_codes[order]
    splits = np.flatnonzero(np.diff(sorted_codes)) + 1
    bounds = np.concatenate(([0], splits, [n_rows]))

    for k in range(len(bounds) - 1):
        s_idx = bounds[k]
        e_idx = bounds[k + 1]
        if s_idx == e_idx:
            continue
        code = int(sorted_codes[s_idx])
        dom_id = code % 10
        if dom_id == 0:
            continue
        hr_h = code // 10
        arr_dict = weather_cache.get((hr_h, dom_id))
        if not arr_dict:
            continue

        row_pos = order[s_idx:e_idx]
        iy = grid_y[row_pos]
        ix = grid_x[row_pos]
        for vname, arr_2d in arr_dict.items():
            raw[vname][row_pos] = arr_2d[iy, ix]

    row_id_arr = (
        df_ais["row_id"].to_numpy(dtype=np.int64)
        if "row_id" in df_ais.columns
        else np.arange(n_rows, dtype=np.int64)
    )
    out = pd.DataFrame({
        "row_id": row_id_arr,
        "mmsi": df_ais["mmsi"].to_numpy(dtype=np.int64),
        "trip_id": df_ais["trip_id"].to_numpy(),
        "base_date_time": df_ais["base_date_time"].to_numpy(),
        "rtma_domain": dom_ids.astype(np.int8),
    })
    out["rtma_TMP_C"] = (raw["TMP"] - KELVIN_OFFSET).astype(np.float32)
    out["rtma_DPT_C"] = (raw["DPT"] - KELVIN_OFFSET).astype(np.float32)
    out["rtma_PRES_hpa"] = (raw["PRES"] * PA_TO_HPA).astype(np.float32)
    out["rtma_VIS_nm"] = (raw["VIS"] * METERS_TO_NM).astype(np.float32)
    out["rtma_WIND_knots"] = (raw["WIND"] * MS_TO_KNOTS).astype(np.float32)
    out["rtma_GUST_knots"] = (raw["GUST"] * MS_TO_KNOTS).astype(np.float32)

    wdir_rad = np.radians(raw["WDIR"].astype(np.float64))
    wdir_sin = np.sin(wdir_rad).astype(np.float32)
    wdir_cos = np.cos(wdir_rad).astype(np.float32)
    out["rtma_wdir_sin"] = wdir_sin
    out["rtma_wdir_cos"] = wdir_cos

    cog_sin = df_ais["cog_sin"].to_numpy(dtype=np.float32)
    cog_cos = df_ais["cog_cos"].to_numpy(dtype=np.float32)
    wind_kn = out["rtma_WIND_knots"].to_numpy(dtype=np.float32)

    # Thành phần gió tương đối so với hướng di chuyển tàu (Lựa chọn B)
    out["rtma_headwind_knots"] = (
        wind_kn * (wdir_cos * cog_cos + wdir_sin * cog_sin)
    ).astype(np.float32)
    out["rtma_crosswind_knots"] = (
        wind_kn * (wdir_sin * cog_cos - wdir_cos * cog_sin)
    ).astype(np.float32)

    core_valid = (
        (out["rtma_domain"].to_numpy() > 0)
        & np.isfinite(out["rtma_TMP_C"].to_numpy())
        & np.isfinite(out["rtma_WIND_knots"].to_numpy())
        & np.isfinite(out["rtma_PRES_hpa"].to_numpy())
    )
    out["weather_missing_flag"] = np.where(
        core_valid, np.int8(0), np.int8(1)
    )
    return out[COLS_WEATHER_SIDECAR]


def join_weather_with_prep27(
    df_prep27: pd.DataFrame,
    df_sidecar: pd.DataFrame,
    use_row_id: bool = True,
) -> pd.DataFrame:
    """Ghép bảng thời tiết sidecar với Bản 3 (27 cột) an toàn 1-1.

    Args:
        df_prep27 (pd.DataFrame): Dữ liệu hoặc lát batch từ Bản 3.
        df_sidecar (pd.DataFrame): Bảng sidecar 16 cột từ DS317_05.
        use_row_id (bool): True để ghép 1-1 theo row_id và đối chiếu
            khóa nghiệp vụ; False nếu ghép theo bộ 3 khóa nghiệp vụ.

    Returns:
        pd.DataFrame: Dữ liệu 39 cột (27 AIS + 12 thời tiết) trong RAM.
    """
    base_df = df_prep27.copy()
    if "Dist_To_Berth_KM" in base_df.columns:
        base_df = base_df.drop(columns=["Dist_To_Berth_KM"])

    if use_row_id:
        if "row_id" not in base_df.columns:
            base_df["row_id"] = np.arange(len(base_df), dtype=np.int64)
        merged = base_df.merge(
            df_sidecar,
            on=["row_id", "mmsi", "trip_id", "base_date_time"],
            how="left",
            validate="one_to_one",
        )
        return merged.drop(columns=["row_id"])

    # Chống nhân dòng hoặc MergeError nếu dữ liệu gốc trùng timestamp
    dedup_sidecar = df_sidecar.drop(
        columns=["row_id"], errors="ignore"
    ).drop_duplicates(
        subset=["mmsi", "trip_id", "base_date_time"], keep="first"
    )
    return base_df.merge(
        dedup_sidecar,
        on=["mmsi", "trip_id", "base_date_time"],
        how="left",
        validate="many_to_one",
    )


def run_self_verification_tests() -> None:
    """Kiểm thử phép chiếu, đơn vị, schema 16 cột sidecar và ghép 39 cột."""
    hi_gds = {
        "gdt": 10, "radius_m": R_EARTH_DEFAULT_M, "nx": 321, "ny": 225,
        "la1": 18.072699, "lo1": 198.474999, "lad": 20.0,
        "la2": 23.087799, "lo2": 206.130999,
        "dx_m": 2500.0, "dy_m": 2500.0,
    }
    fi2, fj2 = latlon_to_grid_ij(
        np.array([hi_gds["la2"]]), np.array([hi_gds["lo2"]]), hi_gds
    )
    assert abs(fi2[0] - 320.0) < 0.02 and abs(fj2[0] - 224.0) < 0.02

    # Kiểm thử suy luận realtime floor('1h') và cơ chế đợi bản tin
    obs = dt.datetime(2025, 7, 1, 12, 10)
    r_wait = resolve_rtma_inference_hour(
        obs, dt.datetime(2025, 7, 1, 12, 15)
    )
    assert r_wait["target_hour_utc"] == dt.datetime(2025, 7, 1, 12, 0)
    assert r_wait["should_wait"] is True
    assert abs(r_wait["wait_seconds"] - 600.0) < 1e-6
    r_ready = resolve_rtma_inference_hour(
        obs, dt.datetime(2025, 7, 1, 12, 40)
    )
    assert r_ready["should_wait"] is False

    cols_27 = [
        "mmsi", "trip_id", "base_date_time", "longitude", "latitude",
        "sog", "cog", "heading", "cog_sin", "cog_cos", "heading_sin",
        "heading_cos", "status", "vessel_type", "length", "width",
        "draft", "cargo", "berth_lon", "berth_lat", "ata_time",
        "Dist_To_Berth_NM", "Area_Proxy", "Volume_Proxy", "Is_Tanker",
        "imputation_flag", "Target_Hours",
    ]
    tiny_ais = pd.DataFrame({c: [0.0] for c in cols_27})
    tiny_ais["row_id"] = np.int64(0)
    tiny_ais["mmsi"] = np.int64(366000001)
    tiny_ais["trip_id"] = "t1"
    tiny_ais["base_date_time"] = pd.Timestamp("2025-07-01 12:30:00")
    tiny_ais["ata_time"] = pd.Timestamp("2025-07-02 12:30:00")
    tiny_ais["cog_sin"] = np.float32(0.0)
    tiny_ais["cog_cos"] = np.float32(1.0)
    tiny_ais["vessel_type"] = np.int32(70)
    tiny_ais["Is_Tanker"] = np.int32(0)
    tiny_ais["imputation_flag"] = np.int32(0)

    # Tàu đi Bắc (cog=0), gió Bắc 10 m/s (wdir=0) -> headwind=+19.44 kn
    hr_key = int(np.datetime64("2025-07-01T12", "h").astype(np.int64))
    fake_grid = {
        v: np.full((2, 2), np.nan, dtype=np.float32) for v in TARGET_VARS
    }
    fake_grid["TMP"][:] = np.float32(298.15)
    fake_grid["DPT"][:] = np.float32(293.15)
    fake_grid["PRES"][:] = np.float32(101325.0)
    fake_grid["VIS"][:] = np.float32(18520.0)
    fake_grid["WDIR"][:] = np.float32(0.0)
    fake_grid["WIND"][:] = np.float32(10.0)
    fake_grid["GUST"][:] = np.float32(15.0)

    res = extract_and_engineer_weather(
        tiny_ais,
        np.array([1], dtype=np.int8),
        np.array([0], dtype=np.int32),
        np.array([0], dtype=np.int32),
        {(hr_key, 1): fake_grid},
    )
    assert list(res.columns) == COLS_WEATHER_SIDECAR
    assert len(res.columns) == 16
    assert abs(float(res["rtma_TMP_C"].iloc[0]) - 25.0) < 1e-4
    assert abs(float(res["rtma_VIS_nm"].iloc[0]) - 10.0) < 1e-4
    assert abs(float(res["rtma_PRES_hpa"].iloc[0]) - 1013.25) < 1e-2
    assert abs(float(res["rtma_headwind_knots"].iloc[0]) - 19.4384) < 1e-2
    assert abs(float(res["rtma_crosswind_knots"].iloc[0])) < 1e-4
    assert int(res["weather_missing_flag"].iloc[0]) == 0

    joined = join_weather_with_prep27(tiny_ais.drop(columns=["row_id"]), res)
    assert len(joined.columns) == 39
    assert abs(float(joined["rtma_TMP_C"].iloc[0]) - 25.0) < 1e-4

    # Kiểm thử bản ghi OOG giữ nguyên NaN + cờ missing = 1
    res_nan = extract_and_engineer_weather(
        tiny_ais,
        np.array([0], dtype=np.int8),
        np.array([-1], dtype=np.int32),
        np.array([-1], dtype=np.int32),
        {},
    )
    assert np.isnan(res_nan["rtma_TMP_C"].iloc[0])
    assert int(res_nan["weather_missing_flag"].iloc[0]) == 1
    print("[SELF-TEST OK] Tất cả kiểm thử sidecar 16 cột & ghép 39 cột đạt!")

In [ ]:
"""Cell 6: Động cơ Streaming nhẹ & Checkpointing tiết kiệm đĩa."""

DOM_ID_TO_CFG = {
    dom_id: (dom_name, tpl) for dom_name, dom_id, tpl in DOMAIN_PRIORITY
}


def _fetch_needed_weather_hours(
    eccodes_mod: object, needed_pairs: list[tuple[int, int]]
) -> dict:
    """Tải song song danh sách (hour_int, dom_id) cần thiết cho chunk."""
    cache = {}
    if not needed_pairs:
        return cache

    def worker(pair: tuple[int, int]) -> tuple[tuple[int, int], dict | None]:
        hr_h, dom_id = pair
        dom_name, tpl = DOM_ID_TO_CFG[dom_id]
        hr_dt = dt.datetime(1970, 1, 1) + dt.timedelta(hours=int(hr_h))
        arrs = fetch_hour_domain_arrays(eccodes_mod, hr_dt, dom_name, tpl)
        return pair, arrs

    with ThreadPoolExecutor(max_workers=MAX_HTTP_WORKERS) as pool:
        futs = [pool.submit(worker, p) for p in needed_pairs]
        for fut in as_completed(futs):
            pair, arrs = fut.result()
            cache[pair] = arrs
    return cache


def _remove_split_shards(shard_paths: list[str]) -> None:
    """Xóa ngay các shard của split đã gộp xong để tiết kiệm disk."""
    for spath in shard_paths:
        for fpath in (spath, spath + ".done"):
            if os.path.exists(fpath):
                os.remove(fpath)


def process_split_streaming(
    split_name: str,
    eccodes_mod: object,
    domain_gds: dict,
) -> dict:
    """Xử lý streaming Bản 3 (7 cột) -> Bảng thời tiết gắn khóa 16 cột.

    Args:
        split_name (str): 'train' hoặc 'test'.
        eccodes_mod (object): Module eccodes.
        domain_gds (dict): Từ điển tham số GRIB2 Section 3 của 4 miền.

    Returns:
        dict: Thống kê số dòng, tỷ lệ missing thời tiết và đường dẫn output.
    """
    src_path = find_split_path(split_name)
    out_path = os.path.join(OUTPUT_WEATHER_DIR, f"{split_name}.parquet")
    if not os.path.exists(src_path):
        raise FileNotFoundError(
            f"Không tìm thấy tệp đầu vào cho {split_name}: {src_path}"
        )

    pf = pq.ParquetFile(src_path)
    total_rows = 0
    row_offset = 0
    missing_rows = 0
    dom_counts = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}
    shard_paths = []
    hours_processed = 0
    read_cols = list(COLS_AIS_NEEDED)

    for b_idx, batch in enumerate(
        pf.iter_batches(
            batch_size=500_000, columns=read_cols, use_threads=True
        )
    ):
        raw_len = len(batch)
        shard_file = os.path.join(
            SHARD_TMP_DIR, f"{split_name}_b{b_idx:05d}.parquet"
        )
        done_flag = shard_file + ".done"

        if os.path.exists(done_flag) and os.path.exists(shard_file):
            meta = pq.read_table(
                shard_file, columns=["rtma_domain", "weather_missing_flag"]
            )
            n_cur = meta.num_rows
            total_rows += n_cur
            row_offset += raw_len
            miss_arr = meta["weather_missing_flag"].to_numpy()
            missing_rows += int((miss_arr == 1).sum())
            d_arr = meta["rtma_domain"].to_numpy()
            for did in dom_counts:
                dom_counts[did] += int((d_arr == did).sum())
            shard_paths.append(shard_file)
            continue

        df_ais = normalize_ais_input_batch(
            pa.Table.from_batches([batch]), start_row_id=row_offset
        )
        row_offset += raw_len

        if TARGET_MONTHS is not None:
            m_mask = df_ais["base_date_time"].dt.month.isin(TARGET_MONTHS)
            df_ais = df_ais.loc[m_mask].reset_index(drop=True)
            if df_ais.empty:
                continue

        lat = df_ais["latitude"].to_numpy(dtype=np.float64)
        lon = df_ais["longitude"].to_numpy(dtype=np.float64)
        dom_ids, grid_y, grid_x = assign_rtma_domain_and_cells(
            lat, lon, domain_gds
        )

        hours_h = (
            df_ais["base_date_time"]
            .dt.floor("1h")
            .to_numpy()
            .astype("datetime64[h]")
            .astype(np.int64)
        )
        unique_pairs = {
            (int(h), int(d))
            for h, d in zip(hours_h, dom_ids)
            if d > 0
        }
        sorted_pairs = sorted(unique_pairs)
        if SMOKE_TEST_HOURS is not None:
            rem = max(0, SMOKE_TEST_HOURS - hours_processed)
            sorted_pairs = sorted_pairs[:rem]
            hours_processed += len(sorted_pairs)

        w_cache = _fetch_needed_weather_hours(eccodes_mod, sorted_pairs)
        df_sidecar = extract_and_engineer_weather(
            df_ais, dom_ids, grid_y, grid_x, w_cache
        )
        w_cache.clear()

        tbl_side = pa.Table.from_pandas(
            df_sidecar, schema=SCHEMA_WEATHER_SIDECAR, preserve_index=False
        )
        tmp_shard = shard_file + ".writing"
        pq.write_table(tbl_side, tmp_shard, compression="zstd")
        os.replace(tmp_shard, shard_file)
        with open(done_flag, "w", encoding="utf-8") as fp:
            fp.write(str(len(df_sidecar)))

        total_rows += len(df_sidecar)
        missing_rows += int((df_sidecar["weather_missing_flag"] == 1).sum())
        for did in dom_counts:
            dom_counts[did] += int((df_sidecar["rtma_domain"] == did).sum())
        shard_paths.append(shard_file)
        print(
            f"[{split_name}] Batch {b_idx}: {len(df_sidecar):,} rows | "
            f"fetched {len(sorted_pairs)} (hour, domain) grids",
            flush=True,
        )
        hit_limit = (
            SMOKE_TEST_HOURS is not None
            and hours_processed >= SMOKE_TEST_HOURS
        )
        if hit_limit:
            break

    tmp_out = out_path + ".writing"
    writer = pq.ParquetWriter(
        tmp_out, SCHEMA_WEATHER_SIDECAR, compression="zstd"
    )
    try:
        for spath in shard_paths:
            tbl = pq.read_table(spath, schema=SCHEMA_WEATHER_SIDECAR)
            writer.write_table(tbl)
    finally:
        writer.close()
    os.replace(tmp_out, out_path)
    _remove_split_shards(shard_paths)

    return {
        "split": split_name,
        "output_path": out_path,
        "total_rows": total_rows,
        "missing_rows": missing_rows,
        "domain_counts": dom_counts,
    }

In [ ]:
"""Cell 7: Thực thi Self-Test & Pipeline Xuất Sidecar RTMA 2025."""


def cleanup_temp_shards(shard_dir: str = SHARD_TMP_DIR) -> None:
    """Dọn dẹp thư mục shard tạm sau khi hợp nhất Parquet hoàn tất."""
    if os.path.exists(shard_dir):
        shutil.rmtree(shard_dir, ignore_errors=True)


def run_weather_pipeline(execute_full_run: bool = False) -> None:
    """Chạy kiểm thử nội bộ và xuất bảng thời tiết sidecar 16 cột.

    Args:
        execute_full_run (bool): False để chỉ kiểm thử logic & Section 3;
            đặt True trên Kaggle để chạy xuất train/test sidecar 16 cột.
    """
    run_self_verification_tests()
    if not execute_full_run:
        print(
            "[INFO] Sẵn sàng chạy trên Kaggle. Đặt execute_full_run=True "
            "khi gắn tập 03_ban_tien_xu_ly_27_thuoc_tinh."
        )
        return

    eccodes_mod = ensure_eccodes()
    domain_gds = discover_domain_gds()
    for name, cfg in domain_gds.items():
        print(
            f"[GDS {name}] GDT=3.{cfg['gdt']} "
            f"Grid={cfg['nx']}x{cfg['ny']} dx={cfg['dx_m']:.1f}m"
        )

    for split_name in ("train", "test"):
        stats = process_split_streaming(
            split_name, eccodes_mod, domain_gds
        )
        miss_pct = (
            100.0 * stats["missing_rows"] / max(1, stats["total_rows"])
        )
        print(
            f"[DONE {split_name}] rows={stats['total_rows']:,} | "
            f"missing={miss_pct:.3f}% | path={stats['output_path']} | "
            f"domains={stats['domain_counts']}",
            flush=True,
        )

    cleanup_temp_shards()


if __name__ == "__main__":
    run_weather_pipeline(execute_full_run=IS_KAGGLE)

## Từ điển 16 Cột của Bảng Thời tiết Sidecar (`weather_rtma_2025_sidecar`)

| STT | Tên cột | Kiểu | Đơn vị / Giá trị | Vai trò & Mô tả |
| :--- | :--- | :--- | :--- | :--- |
| 1 | `row_id` | `int64` | `0..N-1` | Khóa định vị vị trí dòng gốc trong `train.parquet`/`test.parquet` của Bản 3 (chống nhân bản 1-1) |
| 2 | `mmsi` | `int64` | 9 chữ số | Khóa tàu trùng tên & kiểu với Bản 3 |
| 3 | `trip_id` | `string` | chuỗi | Khóa hành trình trùng tên & kiểu với Bản 3 |
| 4 | `base_date_time` | `timestamp[us]` | UTC | Khóa thời gian trùng tên & kiểu với Bản 3 |
| 5 | `rtma_domain` | `int8` | `0..4` | `0`: OOG/Missing, `1`: `rtma2p5` (CONUS), `2`: `akrtma` (AK), `3`: `hirtma` (HI), `4`: `prrtma` (PR) |
| 6 | `weather_missing_flag` | `int8` | `0` hoặc `1` | Cờ `1` nếu bản ghi nằm ngoài lưới RTMA hoặc thiếu biến thời tiết lõi |
| 7 | `rtma_TMP_C` | `float32` | `°C` | Nhiệt độ không khí 2 m (`TMP - 273.15`) |
| 8 | `rtma_DPT_C` | `float32` | `°C` | Điểm sương 2 m (`DPT - 273.15`) |
| 9 | `rtma_PRES_hpa` | `float32` | `hPa` | Áp suất khí quyển bề mặt (`PRES / 100.0`) |
| 10 | `rtma_VIS_nm` | `float32` | `NM` | Tầm nhìn bề mặt quy đổi sang hải lý (`VIS / 1852.0`) |
| 11 | `rtma_WIND_knots` | `float32` | `knots` | Tốc độ gió duy trì 10 m đồng bộ đơn vị `sog` |
| 12 | `rtma_GUST_knots` | `float32` | `knots` | Tốc độ gió giật 10 m |
| 13 | `rtma_wdir_sin` | `float32` | `[-1, 1]` | `sin(radians(WDIR))` mã hóa tuần hoàn hướng gió thổi tới |
| 14 | `rtma_wdir_cos` | `float32` | `[-1, 1]` | `cos(radians(WDIR))` |
| 15 | `rtma_headwind_knots` | `float32` | `knots` | Thành phần gió cản dọc hướng tàu (`+` ngược mũi cản tốc, `-` xuôi gió) |
| 16 | `rtma_crosswind_knots` | `float32` | `knots` | Thành phần gió ngang mạn tàu (`+` mạn phải, `-` mạn trái) |

### Cách ghép nối ngược với Bản 3 (`03_ban_tien_xu_ly_27_thuoc_tinh`) khi huấn luyện
- **Ghép từng batch trên RAM (khuyên dùng, 0 tốn thêm disk):** Đọc song song từng batch `500,000` dòng của `03_ban_tien_xu_ly_27/{split}.parquet` và `weather_rtma_2025_sidecar/{split}.parquet`, gọi `join_weather_with_prep27(df_batch_27, df_batch_sidecar, use_row_id=True)` để kiểm chứng khớp 1-1 (`row_id`, `mmsi`, `trip_id`, `base_date_time`) trước khi đưa vào LightGBM/XGBoost/PyTorch.
- **Ghép qua DuckDB/SQL bằng khóa nghiệp vụ:** `JOIN` hai bảng trên `("mmsi", "trip_id", "base_date_time")` kèm `row_id` (hoặc loại trùng trước nếu dùng khóa 3 cột).